In [69]:
header = ["execID", "problem", "instance", "executable", "full_instance", "status", "exit_code", "real", "time", "user", "system", "memory"]

In [70]:
import re
import os
import sys
import pandas as pd
from matplotlib import pyplot as plt

df = None
for file in os.listdir("."):
    if not file.endswith(".tsv"):
        continue
    df_curr = pd.read_csv(file, sep="\t", names=header, index_col=False)
    df = pd.concat([df, df_curr], ignore_index=True) if not df is None else df_curr

# Computing Metrics

In [71]:
# solved,par1,par10
import numpy as np


df.loc[:, "solved"] = df["status"] == "complete"
# df.loc[:, "par1"] = np.where(
#     df["status"] == "complete",
#     df["real"],
#     1200
# )
# df.loc[:, "par10"] = np.where(
#     df["status"] == "complete",
#     df["real"],
#     1200*10
# )

df.loc[:, "executable"] = df.loc[:, "executable"].str.replace(f"-plain-","-")
df.loc[:, "executable"] = df.loc[:, "executable"].str.replace(r"amowasp-base$","(amo|eo)wasp-base-lg=py", regex=True)

In [72]:
map_execid_to_description = {
    r"firstSubmission": "sub_enc",
    r"AMO_LE": "le_enc",
    r"EO_LE": "le_enc"
}

for execId, desc in map_execid_to_description.items():
    mask = (df["execID"].str.contains(execId, regex=True)) & ~(df["executable"].str.contains(r"^(?:clingo|wasp)", regex=True))
    df.loc[mask, "executable"] = df.loc[mask, "executable"] + f":{desc}"

In [73]:
df["executable"].unique()

array(['amoclingo-lg=c-r=minfly-l=f-smpc=t:le_enc',
       'amoclingo-lg=c-r=minfly-l=h-smpc=t:le_enc',
       'amoclingo-lg=c-r=minfly-l=t-smpc=t:le_enc',
       'amoclingo-lg=c-r=nomin-l=f-smpc=t:le_enc',
       'amoclingo-lg=c-r=nomin-l=h-smpc=t:le_enc',
       'amoclingo-lg=c-r=nomin-l=t-smpc=t:le_enc',
       'eoclingo-lg=c-r=ijcai-l=f-smpc=t:sub_enc',
       'eoclingo-lg=c-r=minfly-l=f-smpc=t:sub_enc',
       'eoclingo-lg=c-r=minfly-l=h-smpc=t:sub_enc',
       'eoclingo-lg=c-r=minfly-l=t-smpc=t:sub_enc',
       'eoclingo-lg=c-r=nomin-l=f-smpc=t:sub_enc',
       'eoclingo-lg=c-r=nomin-l=h-smpc=t:sub_enc',
       'eoclingo-lg=c-r=nomin-l=t-smpc=t:sub_enc',
       'amoclingo-lg=c-r=nomin-l=f-smpc=t:sub_enc',
       'amoclingo-lg=c-r=nomin-l=t-smpc=t:sub_enc',
       'amoclingo-lg=c-r=nomin-l=h-smpc=t:sub_enc',
       'amoclingo-lg=py-r=nomin-l=f-smpc=t:sub_enc',
       'amoclingo-lg=py-r=nomin-l=t-smpc=t:sub_enc',
       'amoclingo-lg=py-r=nomin-l=h-smpc=t:sub_enc',
       'eoclingo

# Df AMO

In [74]:
df_amo = df.copy()
df_amo.drop(index=df[df["executable"].str.contains(r"^eo|-eo$")].index, inplace=True)
df_amo.loc[:, "executable"] = df_amo["executable"].str.replace("-amo$","", regex=True)
df_amo["problem"].unique()

array(['Knapsack', 'GroupAssignment', 'Nurse', 'GraphColouring'],
      dtype=object)

## Pivot AMO

In [75]:
# pivot_amo = df_amo.pivot_table(index=["executable"],columns=["problem"], values=["solved","par1", "par10"], aggfunc={"solved":"sum", "par1": "mean", "par10": "mean"}, margins=True, margins_name='Total', fill_value=0)
pivot_amo = df_amo.pivot_table(index=["executable"],columns=["problem"], values=["solved"], aggfunc={"solved":"sum"}, margins=True, margins_name='Total', fill_value=-1)
pivot_amo = pivot_amo.reorder_levels([1,0], axis=1).sort_index(axis=1)
pivot_amo = pivot_amo.drop(index="Total")
# pivot_amo.loc[:, "solved"].astype(int)
for col in pivot_amo.columns[(pivot_amo.columns.get_level_values(1) == "solved")]:
    pivot_amo[col] = pivot_amo.loc[:, col].astype(int)

for col in pivot_amo.columns:
    pivot_amo[col] = pivot_amo.loc[:, col].round(2)
pivot_amo

problem,GraphColouring,GroupAssignment,Knapsack,Nurse,Total
,solved,solved,solved,solved,solved
executable,,,,,
(amo|eo)wasp-base-lg=py:sub_enc,-1,-1,-1,0,0
amoclingo-base-lg=c:sub_enc,102,307,77,1,487
amoclingo-base-lg=py:sub_enc,95,141,75,0,311
amoclingo-lg=c-r=ijcai-l=f-smpc=t:sub_enc,120,307,77,2,506
amoclingo-lg=c-r=minfly-l=f-smpc=t:le_enc,-1,302,73,1,376
amoclingo-lg=c-r=minfly-l=f-smpc=t:sub_enc,123,306,75,1,505
amoclingo-lg=c-r=minfly-l=h-smpc=t:le_enc,-1,305,73,0,378
amoclingo-lg=c-r=minfly-l=h-smpc=t:sub_enc,122,310,75,0,507


In [76]:
# lazy=false
# reason=nomin
# lang=cpp
# static_mpc=true

import glob


def create_solvers(base_file: str = "base_amo", pivot = pivot_amo, output_dir = "amoclingo_versions"):
    map_key_param = {
    "l": "lazy",
    "r": "reason",
    "lg": "lang",
    "l": "lazy",
    "smpc": "static_mpc"
    }

    map_value = {
    "t": "true",
    "f": "false",
    "h": "hybrid",
    "c": "cpp",
    }

    dir_solvers = "solver_to_move_to_clown"

    base_path = f"{dir_solvers}/{base_file}.sh"
    with open(base_path, "r") as f:
        base_sh = "".join(f.readlines())

    for solver in glob.glob(f"{dir_solvers}/{output_dir}/*"):
        # print(f"removing: {solver}")
        if os.path.isfile(solver):
            os.remove(solver)

    smpc = "t"
    solvers_to_do = set(re.sub(r":.*","",e) for e in list(pivot.index) if not re.search("(-base-|^clingo|wasp|py)", e))
    solvers_to_do = solvers_to_do.union(e.replace("-smpc=t","-smpc=f") for e in solvers_to_do)
    print(f"solvers_to_do: {solvers_to_do}")
    for s in solvers_to_do:
        s = str(s)
        if re.search("(-base-|^clingo|wasp|py)", s): continue
        s = re.sub(r":.*","",s)
        content_s = str(base_sh)
        for key_value in re.findall(r"-\w+=\w+", s):
            key = re.search(r"-(?P<key>\w+)=(?P<value>\w+)", key_value).group("key")
            value = re.search(r"-(?P<key>\w+)=(?P<value>\w+)", key_value).group("value")
            content_s = re.sub(rf"{map_key_param[key]}=\w+", f"{map_key_param[key]}={map_value.get(value, value)}", content_s)
        if "-smpc" in s: s = f"{s}.bash"
        else : s = f"{s}-smpc={smpc}.bash"
        print(f"Writing: {s}")
        with open(f"{dir_solvers}/{output_dir}/{s}", "w") as f:
            print(f"# s: {s}", file=f)
            print(content_s, file=f)

create_solvers(base_file= "base_amo", pivot = pivot_amo, output_dir = "amoclingo_versions")

solvers_to_do: {'amoclingo-lg=c-r=minfly-l=h-smpc=t', 'amoclingo-lg=c-r=nomin-l=t-smpc=f', 'amoclingo-lg=c-r=minfly-l=h-smpc=f', 'amoclingo-lg=c-r=nomin-l=f-smpc=t', 'amoclingo-lg=c-r=minfly-l=t-smpc=f', 'amoclingo-lg=c-r=ijcai-l=f-smpc=t', 'amoclingo-lg=c-r=minfly-l=f-smpc=t', 'amoclingo-lg=c-r=minfly-l=f-smpc=f', 'amoclingo-lg=c-r=ijcai-l=f-smpc=f', 'amoclingo-lg=c-r=nomin-l=h-smpc=t', 'amoclingo-lg=c-r=minfly-l=t-smpc=t', 'amoclingo-lg=c-r=nomin-l=h-smpc=f', 'amoclingo-lg=c-r=nomin-l=f-smpc=f', 'amoclingo-lg=c-r=nomin-l=t-smpc=t'}
Writing: amoclingo-lg=c-r=minfly-l=h-smpc=t.bash
Writing: amoclingo-lg=c-r=nomin-l=t-smpc=f.bash
Writing: amoclingo-lg=c-r=minfly-l=h-smpc=f.bash
Writing: amoclingo-lg=c-r=nomin-l=f-smpc=t.bash
Writing: amoclingo-lg=c-r=minfly-l=t-smpc=f.bash
Writing: amoclingo-lg=c-r=ijcai-l=f-smpc=t.bash
Writing: amoclingo-lg=c-r=minfly-l=f-smpc=t.bash
Writing: amoclingo-lg=c-r=minfly-l=f-smpc=f.bash
Writing: amoclingo-lg=c-r=ijcai-l=f-smpc=f.bash
Writing: amoclingo-lg=c

In [77]:
def create_catcus_df(df_input: pd.DataFrame) -> pd.DataFrame:
    df_catcus = pd.DataFrame()
    for solver in df_input["executable"].unique():
        df_catcus[solver] = df_input[(df_input["executable"] == solver) & (df_input["solved"])]["real"].sort_values().reset_index(drop=True)
    return df_catcus

In [78]:
df_cactus_amo = create_catcus_df(df_amo)
df_cactus_amo.to_csv("plots/catcus_amo.csv")

# Df EO

In [79]:
df_eo = df.copy()
df_eo.drop(index=df[df["executable"].str.contains("(^amo|-amo$)")].index, inplace=True)
df_eo.loc[:, "executable"] = df_eo["executable"].str.replace(r"-eo$","", regex=True)
df_eo

/var/folders/60/msx2m7995xv41v59mx28gt5w0000gn/T/ipykernel_62345/3482083397.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_eo.drop(index=df[df["executable"].str.contains("(^amo|-amo$)")].index, inplace=True)


,execID,problem,instance,executable,full_instance,status,exit_code,real,time,user,system,memory,solved
2730,AIJ@2026-06-07-10-34:firstSubmissionBench,Knapsack,0000-knapsack-10-22288-53089-type1.asp,eoclingo-lg=c-r=ijcai-l=f-smpc=t:sub_enc,results/Knapsack/0000-knapsack-10-22288-53089-...,complete,10,0.423,0.25,0.22,0.03,35.9,True
2731,AIJ@2026-06-07-10-34:firstSubmissionBench,Knapsack,0001-knapsack-10-21042-17651-type1.asp,eoclingo-lg=c-r=ijcai-l=f-smpc=t:sub_enc,results/Knapsack/0001-knapsack-10-21042-17651-...,complete,10,0.685,0.51,0.48,0.03,36.4,True
2732,AIJ@2026-06-07-10-34:firstSubmissionBench,Knapsack,0002-knapsack-10-21756-170352-type1.asp,eoclingo-lg=c-r=ijcai-l=f-smpc=t:sub_enc,results/Knapsack/0002-knapsack-10-21756-170352...,complete,10,0.544,0.37,0.32,0.05,36.8,True
2733,AIJ@2026-06-07-10-34:firstSubmissionBench,Knapsack,0003-knapsack-10-19930-128092-type2.asp,eoclingo-lg=c-r=ijcai-l=f-smpc=t:sub_enc,results/Knapsack/0003-knapsack-10-19930-128092...,complete,10,0.595,0.39,0.36,0.03,36.9,True
2734,AIJ@2026-06-07-10-34:firstSubmissionBench,Knapsack,0004-knapsack-10-21938-2349242-type2.asp,eoclingo-lg=c-r=ijcai-l=f-smpc=t:sub_enc,results/Knapsack/0004-knapsack-10-21938-234924...,complete,20,0.568,0.39,0.35,0.04,36.4,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
29285,AIJ@2026-06-04-09-49:firstSubmissionBench,GroupAssignment,345-group-assignment-250-35-middle.asp,eowasp-base-lg=py:sub_enc,results/GroupAssignment/345-group-assignment-2...,outof time,143,1200.881,1200.22,1195.01,5.21,2740.1,False
29286,AIJ@2026-06-04-09-49:firstSubmissionBench,GroupAssignment,346-group-assignment-250-35-middle.asp,eowasp-base-lg=py:sub_enc,results/GroupAssignment/346-group-assignment-2...,outof time,143,1201.541,1201.05,1197.17,3.88,2070.8,False
29287,AIJ@2026-06-04-09-49:firstSubmissionBench,GroupAssignment,347-group-assignment-250-35-middle.asp,eowasp-base-lg=py:sub_enc,results/GroupAssignment/347-group-assignment-2...,outof time,143,1201.562,1200.56,1195.78,4.78,2317.0,False
29288,AIJ@2026-06-04-09-49:firstSubmissionBench,GroupAssignment,348-group-assignment-250-35-punsat.asp,eowasp-base-lg=py:sub_enc,results/GroupAssignment/348-group-assignment-2...,outof time,143,1201.375,1200.98,1196.50,4.48,2626.3,False


In [80]:
df_cactus_eo = create_catcus_df(df_eo)
df_cactus_eo.to_csv("plots/catcus_eo.csv")

## Pivot EO

In [81]:
# pivot_eo = df_eo.pivot_table(index=["executable"],columns=["problem"], values=["solved","par1", "par10"], aggfunc={"solved":"sum", "par1": "mean", "par10": "mean"}, margins=True, margins_name='Total', fill_value=0)
pivot_eo = df_eo.pivot_table(index=["executable"],columns=["problem"], values=["solved"], aggfunc={"solved":"sum"}, margins=True, margins_name='Total', fill_value=-1)
pivot_eo = pivot_eo.reorder_levels([1,0], axis=1).sort_index(axis=1)
pivot_eo = pivot_eo.drop(index="Total")

for col in pivot_eo.columns[(pivot_eo.columns.get_level_values(1) == "solved")]:
    pivot_eo[col] = pivot_eo.loc[:, col].astype(int)
pivot_eo

problem,GraphColouring,GroupAssignment,Knapsack,Nurse,Total
,solved,solved,solved,solved,solved
executable,,,,,
(amo|eo)wasp-base-lg=py:sub_enc,-1,-1,-1,0,0
clingo,95,350,47,4,496
eoclingo-base-lg=c:sub_enc,101,350,77,1,529
eoclingo-base-lg=py:sub_enc,79,280,74,0,433
eoclingo-lg=c-r=ijcai-l=f-smpc=t:sub_enc,153,350,77,1,581
eoclingo-lg=c-r=minfly-l=f-smpc=t:sub_enc,131,350,74,0,555
eoclingo-lg=c-r=minfly-l=h-smpc=t:sub_enc,132,350,74,0,556
eoclingo-lg=c-r=minfly-l=t-smpc=t:sub_enc,169,350,75,1,595


In [82]:
create_solvers(base_file= "base_eo", pivot = pivot_eo, output_dir = "eoclingo_versions")

solvers_to_do: {'eoclingo-lg=c-r=minfly-l=h-smpc=t', 'eoclingo-lg=c-r=ijcai-l=f-smpc=t', 'eoclingo-lg=c-r=minfly-l=f-smpc=f', 'eoclingo-lg=c-r=minfly-l=f-smpc=t', 'eoclingo-lg=c-r=nomin-l=f-smpc=t', 'eoclingo-lg=c-r=nomin-l=f-smpc=f', 'eoclingo-lg=c-r=nomin-l=t-smpc=t', 'eoclingo-lg=c-r=nomin-l=h-smpc=f', 'eoclingo-lg=c-r=nomin-l=h-smpc=t', 'eoclingo-lg=c-r=nomin-l=t-smpc=f', 'eoclingo-lg=c-r=minfly-l=t-smpc=t', 'eoclingo-lg=c-r=minfly-l=h-smpc=f', 'eoclingo-lg=c-r=minfly-l=t-smpc=f', 'eoclingo-lg=c-r=ijcai-l=f-smpc=f'}
Writing: eoclingo-lg=c-r=minfly-l=h-smpc=t.bash
Writing: eoclingo-lg=c-r=ijcai-l=f-smpc=t.bash
Writing: eoclingo-lg=c-r=minfly-l=f-smpc=f.bash
Writing: eoclingo-lg=c-r=minfly-l=f-smpc=t.bash
Writing: eoclingo-lg=c-r=nomin-l=f-smpc=t.bash
Writing: eoclingo-lg=c-r=nomin-l=f-smpc=f.bash
Writing: eoclingo-lg=c-r=nomin-l=t-smpc=t.bash
Writing: eoclingo-lg=c-r=nomin-l=h-smpc=f.bash
Writing: eoclingo-lg=c-r=nomin-l=h-smpc=t.bash
Writing: eoclingo-lg=c-r=nomin-l=t-smpc=f.bash
W

# Latex Functions

In [83]:
import subprocess
from pathlib import Path
import shutil

def create_latex_visualization(latex_input: str, output_dir: str, output_name, tex_name: str = "main"):
    latex = rf"""
            \documentclass{{article}}
            \usepackage{{booktabs}}
            \usepackage{{multirow}}
            \pagestyle{{empty}}
            \begin{{document}}
            {latex_input}
            \end{{document}}
            """
    output_path = Path(f"{output_dir}/{output_name}")
    output_path.mkdir(parents=True, exist_ok=True)
    tex_name = f"{tex_name}.tex"
    tex_file = output_path / tex_name
    tex_file.write_text(latex)
    subprocess.run(
        ["/Library/TeX/texbin/pdflatex", tex_name],
        cwd=output_path
    )

## Cactus

In [84]:
def create_cactus_latex(file: str, legend_style: str = "at={(1.25,1.0)},anchor=north,fill=none", xmin=0, xmax=None, ymin=0, ymax=1200, subset_solvers = None):
    
    df_cac = pd.read_csv(file, sep=",", header="infer")
    def get_style_plot(solver: str):
        style = {}

        if re.search("amoclingo",solver):
            style["color"] = "blue"
        elif re.search("amowasp", solver):
            style["color"] = "yellow"
        elif re.search("^clingo", solver):
            style["color"] = "red"
        elif re.search("^wasp", solver):
            style["color"] = "green"
        else:
            raise Exception(f"Invalid Solver: {solver}")
        
        if re.search("amoclingo",solver):
            style["mark"] = "o"
        elif re.search("amowasp", solver):
            style["mark"] = "o"
        elif re.search("^clingo", solver):
            style["mark"] = "+"
        elif re.search("^wasp", solver):
            style["mark"] = "+"
        else:
            raise Exception(f"Invalid Solver: {solver}")
        
        return style 
    
    plot_lines = []
    columns = list(df_cac.columns)
    for i in range(1, len(columns)):
        solver = columns[i]
        if not subset_solvers is None and not solver in subset_solvers: continue
        plot_line = []
        style = get_style_plot(solver)
        plot_line = f"""
            \\addplot [mark size=2pt, color={style['color']}, mark={style['mark']}] [unbounded coords=jump] table[col sep=comma, y index={i}] {{./{file}}};
            \\addlegendentry{{{solver}}}
            """
        plot_lines.append(plot_line)

    xmax_str = f"xmax={xmax}" if xmax else ""
    tex_catctus = f"""  
    \\begin{{tikzpicture}}[scale=0.7]
        \\pgfkeys{{/pgf/number format/set thousands separator = {{}}}}
        \\begin{{axis}}[
        scale only axis
        , xlabel={{Solved instances}}
        , ylabel={{Time (s)}}    
        , xmin=0, {xmax_str}
        , ymin=0, ymax=1220
        , legend style={legend_style}
        , legend columns=1
        , width=0.65\\textwidth
        , height=0.40\\textwidth
        , major tick length=2pt
        , title= {{Catctus Plot}}
        ]

        {'\n'.join(plot_lines)}            

        \\end{{axis}}
    \\end{{tikzpicture}}%
    """
    return tex_catctus

catcus_latex_amo = create_cactus_latex("plots/catcus_amo.csv", subset_solvers=["clingo", "wasp", "amoclingo-lg=c-r=minfly-l=t"])
print(catcus_latex_amo)

  
    \begin{tikzpicture}[scale=0.7]
        \pgfkeys{/pgf/number format/set thousands separator = {}}
        \begin{axis}[
        scale only axis
        , xlabel={Solved instances}
        , ylabel={Time (s)}    
        , xmin=0, 
        , ymin=0, ymax=1220
        , legend style=at={(1.25,1.0)},anchor=north,fill=none
        , legend columns=1
        , width=0.65\textwidth
        , height=0.40\textwidth
        , major tick length=2pt
        , title= {Catctus Plot}
        ]

        
            \addplot [mark size=2pt, color=red, mark=+] [unbounded coords=jump] table[col sep=comma, y index=13] {./plots/catcus_amo.csv};
            \addlegendentry{clingo}
            

            \addplot [mark size=2pt, color=green, mark=+] [unbounded coords=jump] table[col sep=comma, y index=14] {./plots/catcus_amo.csv};
            \addlegendentry{wasp}
                        

        \end{axis}
    \end{tikzpicture}%
    


## Tables

In [85]:
def get_pivot_latex(pivot: pd.DataFrame):
    pivot_latex = pivot.copy()
    pivot_latex.index = pivot_latex.index.str.replace("_", "\\_", regex=False)
    for c in pivot_latex.columns:
        maxVal = pivot_latex[c].max()
        pivot_latex[c] = pivot_latex[c].astype(object)
        for r in pivot_latex.index:
            v = pivot_latex.at[r, c]
            if v == maxVal:
                pivot_latex.at[r, c] = rf"\textbf{{{v}}}"
            else:
                pivot_latex.at[r, c] = str(v)
    display(pivot_latex)
    latex = pivot_latex.to_latex(
        multicolumn=True,           
        multicolumn_format='c',     
        multirow=True,              
        bold_rows=False,    
        na_rep='-',                 
        label="tab:results",
        position="t!",
        escape=False,               
    )
    return latex

table_amo_res = get_pivot_latex(pivot_amo)
print(table_amo_res)
create_latex_visualization(table_amo_res, "tables", "table_amo")

problem,GraphColouring,GroupAssignment,Knapsack,Nurse,Total
,solved,solved,solved,solved,solved
executable,,,,,
(amo|eo)wasp-base-lg=py:sub\_enc,-1,-1,-1,0,0
amoclingo-base-lg=c:sub\_enc,102,307,\textbf{77},1,487
amoclingo-base-lg=py:sub\_enc,95,141,75,0,311
amoclingo-lg=c-r=ijcai-l=f-smpc=t:sub\_enc,120,307,\textbf{77},2,506
amoclingo-lg=c-r=minfly-l=f-smpc=t:le\_enc,-1,302,73,1,376
amoclingo-lg=c-r=minfly-l=f-smpc=t:sub\_enc,123,306,75,1,505
amoclingo-lg=c-r=minfly-l=h-smpc=t:le\_enc,-1,305,73,0,378
amoclingo-lg=c-r=minfly-l=h-smpc=t:sub\_enc,122,310,75,0,507


\begin{table}[t!]
\label{tab:results}
\begin{tabular}{llllll}
\toprule
problem & GraphColouring & GroupAssignment & Knapsack & Nurse & Total \\
 & solved & solved & solved & solved & solved \\
executable &  &  &  &  &  \\
\midrule
(amo|eo)wasp-base-lg=py:sub\_enc & -1 & -1 & -1 & 0 & 0 \\
amoclingo-base-lg=c:sub\_enc & 102 & 307 & \textbf{77} & 1 & 487 \\
amoclingo-base-lg=py:sub\_enc & 95 & 141 & 75 & 0 & 311 \\
amoclingo-lg=c-r=ijcai-l=f-smpc=t:sub\_enc & 120 & 307 & \textbf{77} & 2 & 506 \\
amoclingo-lg=c-r=minfly-l=f-smpc=t:le\_enc & -1 & 302 & 73 & 1 & 376 \\
amoclingo-lg=c-r=minfly-l=f-smpc=t:sub\_enc & 123 & 306 & 75 & 1 & 505 \\
amoclingo-lg=c-r=minfly-l=h-smpc=t:le\_enc & -1 & 305 & 73 & 0 & 378 \\
amoclingo-lg=c-r=minfly-l=h-smpc=t:sub\_enc & 122 & 310 & 75 & 0 & 507 \\
amoclingo-lg=c-r=minfly-l=t-smpc=t:le\_enc & -1 & 312 & 75 & 1 & 388 \\
amoclingo-lg=c-r=minfly-l=t-smpc=t:sub\_enc & \textbf{179} & \textbf{314} & 75 & 1 & \textbf{569} \\
amoclingo-lg=c-r=nomin-l=f-smpc=t:le